In [3]:
# ---------------------------------------------------------
# ClimaCare UAE AI
# External Temporal Validation
# Dubai, UAE
# ---------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np
import requests

print("ClimaCare UAE AI — External Validation Started")
print("Location: Dubai, UAE")
print("Purpose: Test the model on data AFTER the original dataset")

ClimaCare UAE AI — External Validation Started
Location: Dubai, UAE
Purpose: Test the model on data AFTER the original dataset


In [4]:
# ---------------------------------------------------------
# Check original Dubai dataset location information
# ---------------------------------------------------------

city_info_path = Path(
    "../data/raw/dubai-ae-292223/city_info.csv"
)

city_info = pd.read_csv(city_info_path)

print("Original Dubai dataset location information:")
print("-" * 55)

display(city_info)

print("\nColumns:")
print(city_info.columns.tolist())

Original Dubai dataset location information:
-------------------------------------------------------


,geoname_id,city_name,country_code,admin1,admin2,feature_code,latitude,longitude,population
0,292223,Dubai,AE,3,NaN,PPLA,25.07725,55.30927,3790000



Columns:
['geoname_id', 'city_name', 'country_code', 'admin1', 'admin2', 'feature_code', 'latitude', 'longitude', 'population']


In [5]:
# ---------------------------------------------------------
# Step 32
# Download NEW Dubai Air-Quality Data
# External Validation Period
# ---------------------------------------------------------

latitude = 25.07725
longitude = 55.30927

start_date = "2026-02-19"
end_date = "2026-08-16"

air_quality_url = (
    "https://air-quality-api.open-meteo.com/v1/air-quality"
)

air_quality_params = {
    "latitude": latitude,
    "longitude": longitude,

    "hourly": [
        "pm10",
        "pm2_5",
        "carbon_monoxide",
        "nitrogen_dioxide",
        "sulphur_dioxide",
        "ozone",
        "aerosol_optical_depth",
        "dust",
        "uv_index",
        "us_aqi",
        "european_aqi"
    ],

    "start_date": start_date,
    "end_date": end_date,

    "timezone": "Asia/Dubai",

    # Dubai uses the global CAMS domain
    "domains": "cams_global"
}

print("Requesting new Dubai air-quality data...")
print("Period:", start_date, "to", end_date)

response = requests.get(
    air_quality_url,
    params=air_quality_params,
    timeout=60
)

print("HTTP Status:", response.status_code)

response.raise_for_status()

air_quality_json = response.json()

print("✅ Air-quality data downloaded successfully!")

print("\nReturned coordinates:")
print("Latitude :", air_quality_json.get("latitude"))
print("Longitude:", air_quality_json.get("longitude"))

print("\nAvailable hourly variables:")
print(list(air_quality_json["hourly"].keys()))

Requesting new Dubai air-quality data...
Period: 2026-02-19 to 2026-08-16
HTTP Status: 200
✅ Air-quality data downloaded successfully!

Returned coordinates:
Latitude : 25.099998
Longitude: 55.300003

Available hourly variables:
['time', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index', 'us_aqi', 'european_aqi']


In [6]:
# ---------------------------------------------------------
# Step 33
# Convert External Dubai Air Quality Data
# Hourly -> Daily
# ---------------------------------------------------------

print("Converting hourly air-quality data to daily format...")

# Convert JSON hourly section into DataFrame
air_hourly_df = pd.DataFrame(
    air_quality_json["hourly"]
)

# Convert time column
air_hourly_df["time"] = pd.to_datetime(
    air_hourly_df["time"]
)

# Variables required by ClimaCare
air_columns = [
    "pm10",
    "pm2_5",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "aerosol_optical_depth",
    "dust",
    "uv_index",
    "us_aqi",
    "european_aqi"
]

# Make sure environmental columns are numeric
for column in air_columns:
    air_hourly_df[column] = pd.to_numeric(
        air_hourly_df[column],
        errors="coerce"
    )

# Create date column
air_hourly_df["date"] = (
    air_hourly_df["time"].dt.date
)

print("\nHOURLY DATA")
print("-" * 50)

print("Shape:", air_hourly_df.shape)

print(
    "Time range:",
    air_hourly_df["time"].min(),
    "to",
    air_hourly_df["time"].max()
)

print("\nMissing hourly values:")
print(
    air_hourly_df[air_columns]
    .isnull()
    .sum()
)


# ---------------------------------------------------------
# Aggregate hourly values into daily averages
# ---------------------------------------------------------

external_air_daily = (
    air_hourly_df
    .groupby("date", as_index=False)[air_columns]
    .mean()
)

# Convert date back to pandas datetime
external_air_daily["date"] = pd.to_datetime(
    external_air_daily["date"]
)

print("\nDAILY EXTERNAL AIR-QUALITY DATA")
print("-" * 50)

print("Shape:", external_air_daily.shape)

print(
    "Date range:",
    external_air_daily["date"].min(),
    "to",
    external_air_daily["date"].max()
)

print("\nMissing daily values:")
print(
    external_air_daily
    .isnull()
    .sum()
)

display(external_air_daily.head())

Converting hourly air-quality data to daily format...

HOURLY DATA
--------------------------------------------------
Shape: (4296, 13)
Time range: 2026-02-19 00:00:00 to 2026-08-16 23:00:00

Missing hourly values:
pm10                     0
pm2_5                    0
carbon_monoxide          0
nitrogen_dioxide         0
sulphur_dioxide          0
ozone                    0
aerosol_optical_depth    0
dust                     0
uv_index                 0
us_aqi                   0
european_aqi             0
dtype: int64

DAILY EXTERNAL AIR-QUALITY DATA
--------------------------------------------------
Shape: (179, 12)
Date range: 2026-02-19 00:00:00 to 2026-08-16 00:00:00

Missing daily values:
date                     0
pm10                     0
pm2_5                    0
carbon_monoxide          0
nitrogen_dioxide         0
sulphur_dioxide          0
ozone                    0
aerosol_optical_depth    0
dust                     0
uv_index                 0
us_aqi                   0

,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi
0,2026-02-19,67.050000,40.116667,315.125000,58.666667,26.041667,50.208333,0.170000,46.625000,1.556250,118.875000,74.375000
1,2026-02-20,51.841667,35.641667,397.916667,56.258333,24.212500,55.500000,0.150000,28.083333,1.550000,107.250000,70.791667
2,2026-02-21,52.216667,38.329167,410.833333,56.429167,29.829167,61.500000,0.152500,23.583333,1.652083,103.208333,69.375000
3,2026-02-22,56.837500,36.429167,596.708333,28.987500,19.612500,117.291667,0.179583,29.458333,1.614583,99.500000,67.583333
4,2026-02-23,85.325000,50.283333,606.041667,33.933333,23.525000,120.625000,0.281250,56.416667,1.618750,123.666667,75.875000


In [7]:
# ---------------------------------------------------------
# Step 34
# Download Weather Data for External Validation
# Dubai, UAE
# ---------------------------------------------------------

weather_url = "https://archive-api.open-meteo.com/v1/archive"

weather_params = {
    "latitude": latitude,
    "longitude": longitude,

    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m"
    ],

    "start_date": start_date,
    "end_date": end_date,

    "timezone": "Asia/Dubai",
    "wind_speed_unit": "kmh"
}

print("Requesting Dubai weather data...")
print("Period:", start_date, "to", end_date)

weather_response = requests.get(
    weather_url,
    params=weather_params,
    timeout=60
)

print("HTTP Status:", weather_response.status_code)

weather_response.raise_for_status()

weather_json = weather_response.json()

print("✅ Weather data downloaded successfully!")

print("\nReturned coordinates:")
print("Latitude :", weather_json.get("latitude"))
print("Longitude:", weather_json.get("longitude"))

print("\nAvailable hourly weather variables:")
print(list(weather_json["hourly"].keys()))

Requesting Dubai weather data...
Period: 2026-02-19 to 2026-08-16
HTTP Status: 200
✅ Weather data downloaded successfully!

Returned coordinates:
Latitude : 25.06151
Longitude: 55.280174

Available hourly weather variables:
['time', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m']


In [8]:
# ---------------------------------------------------------
# Step 35
# Convert External Dubai Weather Data
# Hourly -> Daily
# ---------------------------------------------------------

print("Converting hourly weather data to daily format...")

# Convert API hourly data into DataFrame
weather_hourly_df = pd.DataFrame(
    weather_json["hourly"]
)

# Convert time column
weather_hourly_df["time"] = pd.to_datetime(
    weather_hourly_df["time"]
)

# Weather variables required by ClimaCare
weather_columns = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m"
]

# Make sure values are numeric
for column in weather_columns:
    weather_hourly_df[column] = pd.to_numeric(
        weather_hourly_df[column],
        errors="coerce"
    )

# Create daily date column
weather_hourly_df["date"] = (
    weather_hourly_df["time"].dt.date
)

print("\nHOURLY WEATHER DATA")
print("-" * 50)

print("Shape:", weather_hourly_df.shape)

print(
    "Time range:",
    weather_hourly_df["time"].min(),
    "to",
    weather_hourly_df["time"].max()
)

print("\nMissing hourly weather values:")
print(
    weather_hourly_df[weather_columns]
    .isnull()
    .sum()
)

# ---------------------------------------------------------
# Convert hourly data to daily averages
# ---------------------------------------------------------

external_weather_daily = (
    weather_hourly_df
    .groupby("date", as_index=False)[weather_columns]
    .mean()
)

# Rename to match original ClimaCare dataset
external_weather_daily = external_weather_daily.rename(
    columns={
        "temperature_2m": "temperature",
        "relative_humidity_2m": "humidity",
        "wind_speed_10m": "wind_speed"
    }
)

external_weather_daily["date"] = pd.to_datetime(
    external_weather_daily["date"]
)

print("\nDAILY EXTERNAL WEATHER DATA")
print("-" * 50)

print("Shape:", external_weather_daily.shape)

print(
    "Date range:",
    external_weather_daily["date"].min(),
    "to",
    external_weather_daily["date"].max()
)

print("\nMissing daily weather values:")
print(
    external_weather_daily
    .isnull()
    .sum()
)

display(external_weather_daily.head())

Converting hourly weather data to daily format...

HOURLY WEATHER DATA
--------------------------------------------------
Shape: (4296, 5)
Time range: 2026-02-19 00:00:00 to 2026-08-16 23:00:00

Missing hourly weather values:
temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
dtype: int64

DAILY EXTERNAL WEATHER DATA
--------------------------------------------------
Shape: (179, 4)
Date range: 2026-02-19 00:00:00 to 2026-08-16 00:00:00

Missing daily weather values:
date           0
temperature    0
humidity       0
wind_speed     0
dtype: int64


,date,temperature,humidity,wind_speed
0,2026-02-19,22.475000,49.791667,9.391667
1,2026-02-20,22.170833,50.083333,9.708333
2,2026-02-21,22.525000,58.208333,9.075000
3,2026-02-22,22.183333,73.916667,9.250000
4,2026-02-23,21.483333,81.166667,8.979167


In [9]:
# ---------------------------------------------------------
# Step 36
# Merge External Air Quality + Weather Data
# ---------------------------------------------------------

print("Merging external Dubai air-quality and weather data...")

# Merge using date
external_climacare = pd.merge(
    external_air_daily,
    external_weather_daily,
    on="date",
    how="inner"
)

# Sort chronologically
external_climacare = (
    external_climacare
    .sort_values("date")
    .reset_index(drop=True)
)

print("\nEXTERNAL CLIMACARE DATASET")
print("-" * 55)

print("Shape:", external_climacare.shape)

print(
    "Date range:",
    external_climacare["date"].min(),
    "to",
    external_climacare["date"].max()
)

print("\nColumns:")
print(external_climacare.columns.tolist())

print("\nMissing values:")
print(external_climacare.isnull().sum())

print("\nTotal missing values:")
print(external_climacare.isnull().sum().sum())

display(external_climacare.head())

Merging external Dubai air-quality and weather data...

EXTERNAL CLIMACARE DATASET
-------------------------------------------------------
Shape: (179, 15)
Date range: 2026-02-19 00:00:00 to 2026-08-16 00:00:00

Columns:
['date', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index', 'us_aqi', 'european_aqi', 'temperature', 'humidity', 'wind_speed']

Missing values:
date                     0
pm10                     0
pm2_5                    0
carbon_monoxide          0
nitrogen_dioxide         0
sulphur_dioxide          0
ozone                    0
aerosol_optical_depth    0
dust                     0
uv_index                 0
us_aqi                   0
european_aqi             0
temperature              0
humidity                 0
wind_speed               0
dtype: int64

Total missing values:
0


,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,aerosol_optical_depth,dust,uv_index,us_aqi,european_aqi,temperature,humidity,wind_speed
0,2026-02-19,67.050000,40.116667,315.125000,58.666667,26.041667,50.208333,0.170000,46.625000,1.556250,118.875000,74.375000,22.475000,49.791667,9.391667
1,2026-02-20,51.841667,35.641667,397.916667,56.258333,24.212500,55.500000,0.150000,28.083333,1.550000,107.250000,70.791667,22.170833,50.083333,9.708333
2,2026-02-21,52.216667,38.329167,410.833333,56.429167,29.829167,61.500000,0.152500,23.583333,1.652083,103.208333,69.375000,22.525000,58.208333,9.075000
3,2026-02-22,56.837500,36.429167,596.708333,28.987500,19.612500,117.291667,0.179583,29.458333,1.614583,99.500000,67.583333,22.183333,73.916667,9.250000
4,2026-02-23,85.325000,50.283333,606.041667,33.933333,23.525000,120.625000,0.281250,56.416667,1.618750,123.666667,75.875000,21.483333,81.166667,8.979167


In [10]:
# ---------------------------------------------------------
# Step 37
# Connect Historical + External Dubai Data
# For Correct Lag / Rolling Feature Creation
# ---------------------------------------------------------

print("Preparing continuous Dubai timeline...")

# Load original cleaned historical dataset
historical_path = Path(
    "../data/processed/climacare_clean.csv"
)

historical_clean = pd.read_csv(historical_path)

# Convert date to datetime
historical_clean["date"] = pd.to_datetime(
    historical_clean["date"]
)

print("\nHISTORICAL DATA")
print("-" * 50)

print("Shape:", historical_clean.shape)

print(
    "Date range:",
    historical_clean["date"].min(),
    "to",
    historical_clean["date"].max()
)

# ---------------------------------------------------------
# Combine original history + new external data
# ---------------------------------------------------------

combined_climacare = pd.concat(
    [
        historical_clean,
        external_climacare
    ],
    ignore_index=True
)

# Sort by date
combined_climacare = (
    combined_climacare
    .sort_values("date")
    .reset_index(drop=True)
)

# Check duplicate dates
duplicate_dates = (
    combined_climacare["date"]
    .duplicated()
    .sum()
)

print("\nCOMBINED CONTINUOUS DATA")
print("-" * 50)

print("Shape:", combined_climacare.shape)

print(
    "Date range:",
    combined_climacare["date"].min(),
    "to",
    combined_climacare["date"].max()
)

print("Duplicate dates:", duplicate_dates)

print(
    "Total missing values:",
    combined_climacare.isnull().sum().sum()
)

# ---------------------------------------------------------
# Check the historical -> external transition
# ---------------------------------------------------------

transition_data = combined_climacare[
    (
        combined_climacare["date"] >= "2026-02-15"
    )
    &
    (
        combined_climacare["date"] <= "2026-02-23"
    )
]

print("\nHISTORICAL → EXTERNAL TRANSITION")
print("-" * 50)

display(
    transition_data[
        [
            "date",
            "pm2_5",
            "pm10",
            "us_aqi",
            "temperature",
            "humidity",
            "wind_speed"
        ]
    ]
)

Preparing continuous Dubai timeline...

HISTORICAL DATA
--------------------------------------------------
Shape: (1294, 15)
Date range: 2022-08-05 00:00:00 to 2026-02-18 00:00:00

COMBINED CONTINUOUS DATA
--------------------------------------------------
Shape: (1473, 15)
Date range: 2022-08-05 00:00:00 to 2026-08-16 00:00:00
Duplicate dates: 0
Total missing values: 0

HISTORICAL → EXTERNAL TRANSITION
--------------------------------------------------


,date,pm2_5,pm10,us_aqi,temperature,humidity,wind_speed
1290,2026-02-15,32.270833,42.133333,121.333333,23.016667,45.666667,12.754167
1291,2026-02-16,40.304167,123.129167,92.708333,22.345833,68.250000,16.258333
1292,2026-02-17,50.516667,100.450000,125.208333,22.570833,58.833333,10.933333
1293,2026-02-18,44.241667,75.720833,137.125000,22.112500,44.625000,9.108333
1294,2026-02-19,40.116667,67.050000,118.875000,22.475000,49.791667,9.391667
1295,2026-02-20,35.641667,51.841667,107.250000,22.170833,50.083333,9.708333
1296,2026-02-21,38.329167,52.216667,103.208333,22.525000,58.208333,9.075000
1297,2026-02-22,36.429167,56.837500,99.500000,22.183333,73.916667,9.250000
1298,2026-02-23,50.283333,85.325000,123.666667,21.483333,81.166667,8.979167


In [11]:
# ---------------------------------------------------------
# Step 38
# Recreate Original ClimaCare Feature Engineering
# On Continuous Historical + External Timeline
# ---------------------------------------------------------

print("Recreating original ClimaCare model features...")

feature_df = combined_climacare.copy()

# ---------------------------------------------------------
# 1. Calendar features
# ---------------------------------------------------------

feature_df["year"] = feature_df["date"].dt.year
feature_df["month"] = feature_df["date"].dt.month
feature_df["day"] = feature_df["date"].dt.day
feature_df["day_of_week"] = feature_df["date"].dt.dayofweek


# ---------------------------------------------------------
# 2. Season feature
# Same mapping used in original ClimaCare model
# ---------------------------------------------------------

def get_season(month):

    if month in [12, 1, 2]:
        return "Winter"

    elif month in [3, 4, 5]:
        return "Spring"

    elif month in [6, 7, 8]:
        return "Summer"

    else:
        return "Autumn"


feature_df["season"] = feature_df["month"].apply(
    get_season
)

season_mapping = {
    "Winter": 0,
    "Spring": 1,
    "Summer": 2,
    "Autumn": 3
}

feature_df["season_encoded"] = (
    feature_df["season"]
    .map(season_mapping)
)


# ---------------------------------------------------------
# 3. One-day lag features
# ---------------------------------------------------------

feature_df["pm2_5_lag1"] = (
    feature_df["pm2_5"].shift(1)
)

feature_df["pm10_lag1"] = (
    feature_df["pm10"].shift(1)
)

feature_df["us_aqi_lag1"] = (
    feature_df["us_aqi"].shift(1)
)


# ---------------------------------------------------------
# 4. Shifted 3-day rolling averages
# Only PREVIOUS days are used
# ---------------------------------------------------------

feature_df["pm2_5_roll3"] = (
    feature_df["pm2_5"]
    .shift(1)
    .rolling(window=3)
    .mean()
)

feature_df["pm10_roll3"] = (
    feature_df["pm10"]
    .shift(1)
    .rolling(window=3)
    .mean()
)

feature_df["us_aqi_roll3"] = (
    feature_df["us_aqi"]
    .shift(1)
    .rolling(window=3)
    .mean()
)


# ---------------------------------------------------------
# 5. Interaction features
# ---------------------------------------------------------

feature_df["pm_ratio"] = (
    feature_df["pm2_5"] /
    (feature_df["pm10"] + 1e-6)
)

feature_df["dust_wind"] = (
    feature_df["dust"] /
    (feature_df["wind_speed"] + 1e-6)
)

feature_df["temp_humidity"] = (
    feature_df["temperature"] *
    feature_df["humidity"]
)


# ---------------------------------------------------------
# 6. Next-day AQI target
# ---------------------------------------------------------

feature_df["target_aqi_next_day"] = (
    feature_df["us_aqi"].shift(-1)
)


print("\nFEATURE ENGINEERING COMPLETED")
print("-" * 55)

print("Shape:", feature_df.shape)

print("\nExternal transition check:")

display(
    feature_df.loc[
        feature_df["date"].between(
            "2026-02-18",
            "2026-02-22"
        ),
        [
            "date",
            "pm2_5",
            "pm2_5_lag1",
            "pm2_5_roll3",
            "us_aqi",
            "us_aqi_lag1",
            "us_aqi_roll3",
            "target_aqi_next_day"
        ]
    ]
)

Recreating original ClimaCare model features...

FEATURE ENGINEERING COMPLETED
-------------------------------------------------------
Shape: (1473, 31)

External transition check:


,date,pm2_5,pm2_5_lag1,pm2_5_roll3,us_aqi,us_aqi_lag1,us_aqi_roll3,target_aqi_next_day
1293,2026-02-18,44.241667,50.516667,41.030556,137.125000,125.208333,113.083333,118.875000
1294,2026-02-19,40.116667,44.241667,45.020833,118.875000,137.125000,118.347222,107.250000
1295,2026-02-20,35.641667,40.116667,44.958333,107.250000,118.875000,127.069444,103.208333
1296,2026-02-21,38.329167,35.641667,40.000000,103.208333,107.250000,121.083333,99.500000
1297,2026-02-22,36.429167,38.329167,38.029167,99.500000,103.208333,109.777778,123.666667


In [12]:
# ---------------------------------------------------------
# Step 39
# Prepare Original Training Data + External Validation Data
# Using EXACT Same 28 Features
# ---------------------------------------------------------

print("Preparing exact model feature sets...")

# ---------------------------------------------------------
# 1. Load original feature-engineered dataset
# ---------------------------------------------------------

original_feature_path = Path(
    "../data/processed/climacare_features.csv"
)

original_features = pd.read_csv(
    original_feature_path
)

original_features["date"] = pd.to_datetime(
    original_features["date"]
)

# Create next-day target exactly as in model training
original_features["target_aqi_next_day"] = (
    original_features["us_aqi"].shift(-1)
)

# Last historical day has no next-day target
historical_training = (
    original_features
    .dropna(subset=["target_aqi_next_day"])
    .copy()
)

# ---------------------------------------------------------
# 2. Define EXACT original model features
# ---------------------------------------------------------

feature_columns = [
    column
    for column in historical_training.columns
    if column not in [
        "date",
        "season",
        "target_aqi_next_day"
    ]
]

print("\nNumber of model features:", len(feature_columns))

print("\nModel features:")
for i, column in enumerate(feature_columns, start=1):
    print(f"{i:02d}. {column}")

# ---------------------------------------------------------
# 3. Historical training dataset
# ---------------------------------------------------------

X_historical = historical_training[
    feature_columns
].copy()

y_historical = historical_training[
    "target_aqi_next_day"
].copy()

# ---------------------------------------------------------
# 4. External validation period
#
# Start: first completely new day
# End: 15 Aug because 16 Aug needs 17 Aug AQI as target
# ---------------------------------------------------------

external_validation = feature_df[
    (
        feature_df["date"] >= "2026-02-19"
    )
    &
    (
        feature_df["date"] <= "2026-08-15"
    )
].copy()

X_external = external_validation[
    feature_columns
].copy()

y_external = external_validation[
    "target_aqi_next_day"
].copy()

external_dates = external_validation[
    "date"
].copy()

# ---------------------------------------------------------
# 5. Validation checks
# ---------------------------------------------------------

print("\nHISTORICAL TRAINING SET")
print("-" * 55)

print("Samples:", len(X_historical))

print(
    "Period:",
    historical_training["date"].min(),
    "to",
    historical_training["date"].max()
)

print("Features:", X_historical.shape[1])

print(
    "Missing X values:",
    X_historical.isnull().sum().sum()
)

print(
    "Missing y values:",
    y_historical.isnull().sum()
)


print("\nEXTERNAL VALIDATION SET")
print("-" * 55)

print("Samples:", len(X_external))

print(
    "Period:",
    external_dates.min(),
    "to",
    external_dates.max()
)

print("Features:", X_external.shape[1])

print(
    "Missing X values:",
    X_external.isnull().sum().sum()
)

print(
    "Missing y values:",
    y_external.isnull().sum()
)

print(
    "\nFeature columns identical:",
    list(X_historical.columns)
    ==
    list(X_external.columns)
)

print(
    "Historical/external overlap:",
    historical_training["date"].max()
    >=
    external_dates.min()
)

Preparing exact model feature sets...

Number of model features: 28

Model features:
01. pm10
02. pm2_5
03. carbon_monoxide
04. nitrogen_dioxide
05. sulphur_dioxide
06. ozone
07. aerosol_optical_depth
08. dust
09. uv_index
10. us_aqi
11. european_aqi
12. temperature
13. humidity
14. wind_speed
15. year
16. month
17. day
18. day_of_week
19. pm2_5_lag1
20. pm10_lag1
21. us_aqi_lag1
22. pm2_5_roll3
23. pm10_roll3
24. us_aqi_roll3
25. pm_ratio
26. dust_wind
27. temp_humidity
28. season_encoded

HISTORICAL TRAINING SET
-------------------------------------------------------
Samples: 1290
Period: 2022-08-08 00:00:00 to 2026-02-17 00:00:00
Features: 28
Missing X values: 0
Missing y values: 0

EXTERNAL VALIDATION SET
-------------------------------------------------------
Samples: 178
Period: 2026-02-19 00:00:00 to 2026-08-15 00:00:00
Features: 28
Missing X values: 0
Missing y values: 0

Feature columns identical: True
Historical/external overlap: False


In [13]:
# ---------------------------------------------------------
# Step 40
# ClimaCare UAE AI
# External Temporal Validation
# Final Selected Random Forest
# ---------------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


print("Training final ClimaCare Random Forest...")
print("Training data: historical Dubai period only")
print("Testing data : newer unseen Dubai period")


# ---------------------------------------------------------
# 1. Train our selected Random Forest
# ---------------------------------------------------------

climacare_rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

climacare_rf.fit(
    X_historical,
    y_historical
)


# ---------------------------------------------------------
# 2. Predict the external Dubai period
# ---------------------------------------------------------

external_predictions = climacare_rf.predict(
    X_external
)


# ---------------------------------------------------------
# 3. Regression Evaluation
# ---------------------------------------------------------

external_mae = mean_absolute_error(
    y_external,
    external_predictions
)

external_rmse = np.sqrt(
    mean_squared_error(
        y_external,
        external_predictions
    )
)

external_r2 = r2_score(
    y_external,
    external_predictions
)


print("\nCLIMACARE EXTERNAL TEMPORAL VALIDATION")
print("-" * 60)

print(f"External samples : {len(y_external)}")
print(f"MAE              : {external_mae:.3f}")
print(f"RMSE             : {external_rmse:.3f}")
print(f"R²               : {external_r2:.3f}")


# ---------------------------------------------------------
# 4. Persistence Baseline
#
# Simple comparison:
# tomorrow AQI ≈ today's AQI
# ---------------------------------------------------------

baseline_predictions = (
    external_validation["us_aqi"].values
)

baseline_mae = mean_absolute_error(
    y_external,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_external,
        baseline_predictions
    )
)

baseline_r2 = r2_score(
    y_external,
    baseline_predictions
)


print("\nPERSISTENCE BASELINE")
print("-" * 60)

print(f"MAE  : {baseline_mae:.3f}")
print(f"RMSE : {baseline_rmse:.3f}")
print(f"R²   : {baseline_r2:.3f}")


# ---------------------------------------------------------
# 5. AI Improvement over Baseline
# ---------------------------------------------------------

mae_improvement = (
    (baseline_mae - external_mae)
    / baseline_mae
) * 100


print("\nAI VS BASELINE")
print("-" * 60)

print(f"ClimaCare RF R² : {external_r2:.3f}")
print(f"Baseline R²     : {baseline_r2:.3f}")

print(
    f"MAE improvement over baseline : "
    f"{mae_improvement:.2f}%"
)


# ---------------------------------------------------------
# 6. ClimaCare Early-Warning Evaluation
#
# Actual risky day:
# next-day US AQI > 100
#
# The warning threshold remains FIXED at 90.
# We do NOT retune it using this external dataset.
# ---------------------------------------------------------

CLIMACARE_WARNING_THRESHOLD = 90

actual_risky_days = (
    y_external.values > 100
).astype(int)

predicted_warnings = (
    external_predictions >
    CLIMACARE_WARNING_THRESHOLD
).astype(int)


warning_accuracy = accuracy_score(
    actual_risky_days,
    predicted_warnings
)

warning_precision = precision_score(
    actual_risky_days,
    predicted_warnings,
    zero_division=0
)

warning_recall = recall_score(
    actual_risky_days,
    predicted_warnings,
    zero_division=0
)

warning_f1 = f1_score(
    actual_risky_days,
    predicted_warnings,
    zero_division=0
)


tn, fp, fn, tp = confusion_matrix(
    actual_risky_days,
    predicted_warnings,
    labels=[0, 1]
).ravel()


print("\nCLIMACARE EARLY-WARNING PERFORMANCE")
print("-" * 60)

print(
    f"Warning threshold : "
    f"{CLIMACARE_WARNING_THRESHOLD}"
)

print(
    f"Accuracy  : "
    f"{warning_accuracy * 100:.2f}%"
)

print(
    f"Precision : "
    f"{warning_precision * 100:.2f}%"
)

print(
    f"Recall    : "
    f"{warning_recall * 100:.2f}%"
)

print(
    f"F1 Score  : "
    f"{warning_f1 * 100:.2f}%"
)


print("\nSAFETY OUTCOMES")
print("-" * 60)

print("Correct safe days      :", tn)
print("False warnings         :", fp)
print("Missed risky days      :", fn)
print("Correct risky warnings :", tp)


# ---------------------------------------------------------
# 7. Store External Prediction Results
# ---------------------------------------------------------

external_results = pd.DataFrame({

    "date":
        external_dates.values,

    "current_aqi":
        external_validation["us_aqi"].values,

    "actual_next_day_aqi":
        y_external.values,

    "predicted_next_day_aqi":
        external_predictions

})


external_results["absolute_error"] = np.abs(

    external_results["actual_next_day_aqi"]
    -
    external_results["predicted_next_day_aqi"]

)


# Did ClimaCare issue an early warning?
external_results["early_warning"] = (

    external_results["predicted_next_day_aqi"]
    >
    CLIMACARE_WARNING_THRESHOLD

)


print("\nFIRST 10 EXTERNAL PREDICTIONS")
print("-" * 60)

display(
    external_results.head(10)
)

Training final ClimaCare Random Forest...
Training data: historical Dubai period only
Testing data : newer unseen Dubai period

CLIMACARE EXTERNAL TEMPORAL VALIDATION
------------------------------------------------------------
External samples : 178
MAE              : 9.068
RMSE             : 11.143
R²               : 0.851

PERSISTENCE BASELINE
------------------------------------------------------------
MAE  : 11.643
RMSE : 15.773
R²   : 0.702

AI VS BASELINE
------------------------------------------------------------
ClimaCare RF R² : 0.851
Baseline R²     : 0.702
MAE improvement over baseline : 22.11%

CLIMACARE EARLY-WARNING PERFORMANCE
------------------------------------------------------------
Warning threshold : 90
Accuracy  : 89.89%
Precision : 91.19%
Recall    : 97.32%
F1 Score  : 94.16%

SAFETY OUTCOMES
------------------------------------------------------------
Correct safe days      : 15
False warnings         : 14
Missed risky days      : 4
Correct risky warnings : 14

,date,current_aqi,actual_next_day_aqi,predicted_next_day_aqi,absolute_error,early_warning
0,2026-02-19,118.875000,107.250000,116.605556,9.355556,True
1,2026-02-20,107.250000,103.208333,96.550417,6.657917,True
2,2026-02-21,103.208333,99.500000,110.056111,10.556111,True
3,2026-02-22,99.500000,123.666667,105.716667,17.950000,True
4,2026-02-23,123.666667,142.458333,134.476667,7.981667,True
5,2026-02-24,142.458333,137.500000,138.521528,1.021528,True
6,2026-02-25,137.500000,109.708333,123.188611,13.480278,True
7,2026-02-26,109.708333,82.333333,89.986667,7.653333,False
8,2026-02-27,82.333333,102.875000,87.619028,15.255972,False
9,2026-02-28,102.875000,138.333333,126.985556,11.347778,True


In [14]:
# ---------------------------------------------------------
# Step 41
# Save ClimaCare Production Forecasting Model
# ---------------------------------------------------------

from pathlib import Path
import joblib
import json

print("Preparing ClimaCare production AI model...")

# ---------------------------------------------------------
# Model storage folder
# ---------------------------------------------------------

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "climacare_aqi_forecaster.joblib"
features_path = model_dir / "climacare_feature_schema.json"
metadata_path = model_dir / "climacare_model_metadata.json"


# ---------------------------------------------------------
# 1. Save trained Random Forest
# ---------------------------------------------------------

joblib.dump(
    climacare_rf,
    model_path
)


# ---------------------------------------------------------
# 2. Save exact model feature order
# ---------------------------------------------------------

feature_schema = {
    "feature_count": len(feature_columns),
    "features": feature_columns
}

with open(
    features_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        feature_schema,
        file,
        indent=4
    )


# ---------------------------------------------------------
# 3. Save ClimaCare model metadata
# ---------------------------------------------------------

model_metadata = {

    "project":
        "ClimaCare UAE AI",

    "model_type":
        "RandomForestRegressor",

    "prediction_task":
        "Next-day Dubai US AQI forecasting",

    "location":
        "Dubai, UAE",

    "training_end_date":
        "2026-02-17",

    "external_validation_start":
        "2026-02-19",

    "external_validation_end":
        "2026-08-15",

    "feature_count":
        len(feature_columns),

    "warning_threshold":
        90,

    "risk_definition":
        "Actual next-day US AQI > 100",

    "external_validation": {

        "samples":
            int(len(y_external)),

        "mae":
            float(external_mae),

        "rmse":
            float(external_rmse),

        "r2":
            float(external_r2),

        "baseline_r2":
            float(baseline_r2),

        "mae_improvement_percent":
            float(mae_improvement),

        "warning_precision":
            float(warning_precision),

        "warning_recall":
            float(warning_recall),

        "warning_f1":
            float(warning_f1),

        "missed_risky_days":
            int(fn),

        "false_warnings":
            int(fp)
    }
}


with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        model_metadata,
        file,
        indent=4
    )


# ---------------------------------------------------------
# 4. Save external validation predictions
# ---------------------------------------------------------

results_path = (
    Path("../data/processed")
    / "external_validation_results.csv"
)

external_results.to_csv(
    results_path,
    index=False
)


# ---------------------------------------------------------
# Confirmation
# ---------------------------------------------------------

print("\n✅ CLIMACARE PRODUCTION MODEL SAVED")
print("-" * 60)

print("Model:")
print(model_path)

print("\nFeature schema:")
print(features_path)

print("\nMetadata:")
print(metadata_path)

print("\nExternal validation results:")
print(results_path)

print("\nFeatures saved:", len(feature_columns))
print("External R²:", round(external_r2, 3))
print(
    "External warning recall:",
    f"{warning_recall * 100:.2f}%"
)

Preparing ClimaCare production AI model...

✅ CLIMACARE PRODUCTION MODEL SAVED
------------------------------------------------------------
Model:
../models/climacare_aqi_forecaster.joblib

Feature schema:
../models/climacare_feature_schema.json

Metadata:
../models/climacare_model_metadata.json

External validation results:
../data/processed/external_validation_results.csv

Features saved: 28
External R²: 0.851
External warning recall: 97.32%


In [15]:
# ---------------------------------------------------------
# Step 42
# ClimaCare UAE AI
# Final Deployment Model
#
# IMPORTANT:
# The externally validated model is preserved separately.
# The deployment model is then retrained using all
# currently available labelled Dubai data.
# ---------------------------------------------------------

from pathlib import Path
import shutil
import joblib
import json

from sklearn.ensemble import RandomForestRegressor


print("Preparing ClimaCare deployment model...")


# ---------------------------------------------------------
# 1. Model paths
# ---------------------------------------------------------

model_dir = Path("../models")

current_model_path = (
    model_dir /
    "climacare_aqi_forecaster.joblib"
)

evaluation_model_path = (
    model_dir /
    "climacare_evaluation_model.joblib"
)

deployment_model_path = (
    model_dir /
    "climacare_deployment_model.joblib"
)

deployment_metadata_path = (
    model_dir /
    "climacare_deployment_metadata.json"
)


# ---------------------------------------------------------
# 2. Preserve the externally validated model
# ---------------------------------------------------------

shutil.copy2(
    current_model_path,
    evaluation_model_path
)

print(
    "✅ Externally validated model preserved."
)


# ---------------------------------------------------------
# 3. Prepare ALL currently available labelled data
# ---------------------------------------------------------

deployment_training = feature_df.dropna(
    subset=
        feature_columns
        +
        ["target_aqi_next_day"]
).copy()


X_deployment = deployment_training[
    feature_columns
].copy()

y_deployment = deployment_training[
    "target_aqi_next_day"
].copy()


print("\nDEPLOYMENT TRAINING DATA")
print("-" * 60)

print(
    "Samples:",
    len(X_deployment)
)

print(
    "Period:",
    deployment_training["date"].min(),
    "to",
    deployment_training["date"].max()
)

print(
    "Features:",
    X_deployment.shape[1]
)

print(
    "Missing X:",
    X_deployment.isnull().sum().sum()
)

print(
    "Missing y:",
    y_deployment.isnull().sum()
)


# ---------------------------------------------------------
# 4. Train final deployment Random Forest
# ---------------------------------------------------------

deployment_rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

deployment_rf.fit(
    X_deployment,
    y_deployment
)


# ---------------------------------------------------------
# 5. Save deployment model
# ---------------------------------------------------------

joblib.dump(
    deployment_rf,
    deployment_model_path
)


# Also make this the model used by the future app
joblib.dump(
    deployment_rf,
    current_model_path
)


# ---------------------------------------------------------
# 6. Deployment metadata
# ---------------------------------------------------------

deployment_metadata = {

    "project":
        "ClimaCare UAE AI",

    "model_role":
        "Deployment forecasting model",

    "model_type":
        "RandomForestRegressor",

    "prediction_task":
        "Next-day Dubai US AQI forecasting",

    "location":
        "Dubai, UAE",

    "feature_count":
        len(feature_columns),

    "training_samples":
        int(len(X_deployment)),

    "training_start_date":
        str(
            deployment_training["date"]
            .min()
            .date()
        ),

    "training_end_date":
        str(
            deployment_training["date"]
            .max()
            .date()
        ),

    "warning_threshold":
        90,

    "evaluation_reference": {

        "external_r2":
            float(external_r2),

        "external_mae":
            float(external_mae),

        "external_rmse":
            float(external_rmse),

        "external_warning_recall":
            float(warning_recall),

        "external_warning_f1":
            float(warning_f1),

        "note":
            (
                "These metrics belong to the preserved "
                "evaluation model tested before the "
                "external period was added to training."
            )
    }
}


with open(
    deployment_metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        deployment_metadata,
        file,
        indent=4
    )


# ---------------------------------------------------------
# 7. Final confirmation
# ---------------------------------------------------------

print("\n✅ CLIMACARE DEPLOYMENT MODEL CREATED")
print("-" * 60)

print(
    "Evaluation model:",
    evaluation_model_path
)

print(
    "Deployment model:",
    deployment_model_path
)

print(
    "App model:",
    current_model_path
)

print(
    "Deployment metadata:",
    deployment_metadata_path
)

print(
    "\nDeployment training samples:",
    len(X_deployment)
)

print(
    "Deployment feature count:",
    X_deployment.shape[1]
)

print(
    "\nPreviously validated External R²:",
    round(external_r2, 3)
)

print(
    "Previously validated Warning Recall:",
    f"{warning_recall * 100:.2f}%"
)

Preparing ClimaCare deployment model...
✅ Externally validated model preserved.

DEPLOYMENT TRAINING DATA
------------------------------------------------------------
Samples: 1469
Period: 2022-08-08 00:00:00 to 2026-08-15 00:00:00
Features: 28
Missing X: 0
Missing y: 0

✅ CLIMACARE DEPLOYMENT MODEL CREATED
------------------------------------------------------------
Evaluation model: ../models/climacare_evaluation_model.joblib
Deployment model: ../models/climacare_deployment_model.joblib
App model: ../models/climacare_aqi_forecaster.joblib
Deployment metadata: ../models/climacare_deployment_metadata.json

Deployment training samples: 1469
Deployment feature count: 28

Previously validated External R²: 0.851
Previously validated Warning Recall: 97.32%


In [16]:
# ---------------------------------------------------------
# Step 43
# ClimaCare UAE AI
# Global Explainable AI
# Permutation Importance on External Validation Data
# ---------------------------------------------------------

import joblib
import pandas as pd
import numpy as np

from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score


print("Starting ClimaCare Explainable AI analysis...")


# ---------------------------------------------------------
# 1. Load the preserved evaluation model
# ---------------------------------------------------------

evaluation_model = joblib.load(
    "../models/climacare_evaluation_model.joblib"
)

print("✅ Preserved evaluation model loaded.")


# ---------------------------------------------------------
# 2. Verify that the loaded model still reproduces
#    our external validation result
# ---------------------------------------------------------

verification_predictions = evaluation_model.predict(
    X_external
)

verification_r2 = r2_score(
    y_external,
    verification_predictions
)

print(
    "\nVerification External R²:",
    round(verification_r2, 3)
)


# ---------------------------------------------------------
# 3. Calculate permutation importance
#    using ONLY the unseen external validation period
# ---------------------------------------------------------

print("\nCalculating external permutation importance...")

permutation_result = permutation_importance(
    evaluation_model,
    X_external,
    y_external,
    scoring="r2",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)


# ---------------------------------------------------------
# 4. Create importance table
# ---------------------------------------------------------

importance_df = pd.DataFrame({

    "feature":
        X_external.columns,

    "importance_mean":
        permutation_result.importances_mean,

    "importance_std":
        permutation_result.importances_std

})


importance_df = (
    importance_df
    .sort_values(
        "importance_mean",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nTOP 15 CLIMACARE PREDICTION DRIVERS")
print("-" * 65)

display(
    importance_df.head(15).round(4)
)


# ---------------------------------------------------------
# 5. Show most influential feature
# ---------------------------------------------------------

top_feature = importance_df.iloc[0]

print("\nMOST INFLUENTIAL EXTERNAL FEATURE")
print("-" * 65)

print(
    "Feature:",
    top_feature["feature"]
)

print(
    "Mean importance:",
    round(
        top_feature["importance_mean"],
        4
    )
)

print(
    "Importance variation:",
    round(
        top_feature["importance_std"],
        4
    )
)

Starting ClimaCare Explainable AI analysis...
✅ Preserved evaluation model loaded.

Verification External R²: 0.851

Calculating external permutation importance...

TOP 15 CLIMACARE PREDICTION DRIVERS
-----------------------------------------------------------------


,feature,importance_mean,importance_std
0,pm2_5,1.2789,0.1091
1,ozone,0.0215,0.0088
2,us_aqi,0.0163,0.0031
3,temperature,0.0095,0.0024
4,uv_index,0.0088,0.0027
5,pm10,0.0039,0.0010
6,aerosol_optical_depth,0.0032,0.0009
7,sulphur_dioxide,0.0024,0.0011
8,nitrogen_dioxide,0.0021,0.0014
9,us_aqi_roll3,0.0020,0.0009



MOST INFLUENTIAL EXTERNAL FEATURE
-----------------------------------------------------------------
Feature: pm2_5
Mean importance: 1.2789
Importance variation: 0.1091


In [17]:
# ---------------------------------------------------------
# Step 43
# ClimaCare UAE AI
# Global Explainable AI
# Permutation Importance on External Validation Data
# ---------------------------------------------------------

import joblib
import pandas as pd
import numpy as np

from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score


print("Starting ClimaCare Explainable AI analysis...")


# ---------------------------------------------------------
# 1. Load the preserved evaluation model
# ---------------------------------------------------------

evaluation_model = joblib.load(
    "../models/climacare_evaluation_model.joblib"
)

print("✅ Preserved evaluation model loaded.")


# ---------------------------------------------------------
# 2. Verify that the loaded model still reproduces
#    our external validation result
# ---------------------------------------------------------

verification_predictions = evaluation_model.predict(
    X_external
)

verification_r2 = r2_score(
    y_external,
    verification_predictions
)

print(
    "\nVerification External R²:",
    round(verification_r2, 3)
)


# ---------------------------------------------------------
# 3. Calculate permutation importance
#    using ONLY the unseen external validation period
# ---------------------------------------------------------

print("\nCalculating external permutation importance...")

permutation_result = permutation_importance(
    evaluation_model,
    X_external,
    y_external,
    scoring="r2",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)


# ---------------------------------------------------------
# 4. Create importance table
# ---------------------------------------------------------

importance_df = pd.DataFrame({

    "feature":
        X_external.columns,

    "importance_mean":
        permutation_result.importances_mean,

    "importance_std":
        permutation_result.importances_std

})


importance_df = (
    importance_df
    .sort_values(
        "importance_mean",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nTOP 15 CLIMACARE PREDICTION DRIVERS")
print("-" * 65)

display(
    importance_df.head(15).round(4)
)


# ---------------------------------------------------------
# 5. Show most influential feature
# ---------------------------------------------------------

top_feature = importance_df.iloc[0]

print("\nMOST INFLUENTIAL EXTERNAL FEATURE")
print("-" * 65)

print(
    "Feature:",
    top_feature["feature"]
)

print(
    "Mean importance:",
    round(
        top_feature["importance_mean"],
        4
    )
)

print(
    "Importance variation:",
    round(
        top_feature["importance_std"],
        4
    )
)

Starting ClimaCare Explainable AI analysis...
✅ Preserved evaluation model loaded.

Verification External R²: 0.851

Calculating external permutation importance...

TOP 15 CLIMACARE PREDICTION DRIVERS
-----------------------------------------------------------------


,feature,importance_mean,importance_std
0,pm2_5,1.2789,0.1091
1,ozone,0.0215,0.0088
2,us_aqi,0.0163,0.0031
3,temperature,0.0095,0.0024
4,uv_index,0.0088,0.0027
5,pm10,0.0039,0.0010
6,aerosol_optical_depth,0.0032,0.0009
7,sulphur_dioxide,0.0024,0.0011
8,nitrogen_dioxide,0.0021,0.0014
9,us_aqi_roll3,0.0020,0.0009



MOST INFLUENTIAL EXTERNAL FEATURE
-----------------------------------------------------------------
Feature: pm2_5
Mean importance: 1.2789
Importance variation: 0.1091


In [18]:
# ---------------------------------------------------------
# Step 44
# ClimaCare UAE AI
# Build Human-Readable Explainability Profile
# ---------------------------------------------------------

from pathlib import Path
import pandas as pd
import json


print("Building ClimaCare explainability profile...")


# ---------------------------------------------------------
# 1. Human-readable feature names
# ---------------------------------------------------------

feature_labels = {

    "pm2_5":
        "Fine Particulate Matter (PM2.5)",

    "pm10":
        "Particulate Matter (PM10)",

    "carbon_monoxide":
        "Carbon Monoxide",

    "nitrogen_dioxide":
        "Nitrogen Dioxide",

    "sulphur_dioxide":
        "Sulphur Dioxide",

    "ozone":
        "Ground-Level Ozone",

    "aerosol_optical_depth":
        "Atmospheric Aerosol Load",

    "dust":
        "Dust Concentration",

    "uv_index":
        "UV Index",

    "us_aqi":
        "Current Air Quality Index",

    "european_aqi":
        "European AQI Indicator",

    "temperature":
        "Temperature",

    "humidity":
        "Relative Humidity",

    "wind_speed":
        "Wind Speed",

    "year":
        "Year",

    "month":
        "Month",

    "day":
        "Day of Month",

    "day_of_week":
        "Day of Week",

    "pm2_5_lag1":
        "Previous-Day PM2.5",

    "pm10_lag1":
        "Previous-Day PM10",

    "us_aqi_lag1":
        "Previous-Day AQI",

    "pm2_5_roll3":
        "3-Day PM2.5 Trend",

    "pm10_roll3":
        "3-Day PM10 Trend",

    "us_aqi_roll3":
        "3-Day AQI Trend",

    "pm_ratio":
        "Fine-to-Coarse Particle Ratio",

    "dust_wind":
        "Dust-Wind Interaction",

    "temp_humidity":
        "Temperature-Humidity Interaction",

    "season_encoded":
        "Season"
}


# ---------------------------------------------------------
# 2. Feature groups
# ---------------------------------------------------------

feature_groups = {

    "pm2_5": "Air Pollution",
    "pm10": "Air Pollution",
    "carbon_monoxide": "Air Pollution",
    "nitrogen_dioxide": "Air Pollution",
    "sulphur_dioxide": "Air Pollution",
    "ozone": "Air Pollution",
    "aerosol_optical_depth": "Atmosphere",
    "dust": "Dust / Atmosphere",
    "uv_index": "Climate / Exposure",
    "us_aqi": "Air Quality",
    "european_aqi": "Air Quality",

    "temperature": "Weather",
    "humidity": "Weather",
    "wind_speed": "Weather",

    "year": "Temporal",
    "month": "Temporal",
    "day": "Temporal",
    "day_of_week": "Temporal",
    "season_encoded": "Temporal",

    "pm2_5_lag1": "Historical Pollution",
    "pm10_lag1": "Historical Pollution",
    "us_aqi_lag1": "Historical Air Quality",

    "pm2_5_roll3": "Historical Trend",
    "pm10_roll3": "Historical Trend",
    "us_aqi_roll3": "Historical Trend",

    "pm_ratio": "Engineered Environmental",
    "dust_wind": "Engineered Environmental",
    "temp_humidity": "Engineered Environmental"
}


# ---------------------------------------------------------
# 3. Add readable information to permutation results
# ---------------------------------------------------------

explainability_df = importance_df.copy()

explainability_df["display_name"] = (
    explainability_df["feature"]
    .map(feature_labels)
    .fillna(explainability_df["feature"])
)

explainability_df["feature_group"] = (
    explainability_df["feature"]
    .map(feature_groups)
    .fillna("Other")
)


# ---------------------------------------------------------
# 4. Rank features
# ---------------------------------------------------------

explainability_df["rank"] = (
    range(
        1,
        len(explainability_df) + 1
    )
)

explainability_df = explainability_df[
    [
        "rank",
        "feature",
        "display_name",
        "feature_group",
        "importance_mean",
        "importance_std"
    ]
]


print("\nTOP 10 HUMAN-READABLE MODEL DRIVERS")
print("-" * 70)

display(
    explainability_df
    .head(10)
    .round(4)
)


# ---------------------------------------------------------
# 5. Save full explainability table
# ---------------------------------------------------------

explainability_csv_path = (
    Path("../models")
    /
    "climacare_global_explainability.csv"
)

explainability_df.to_csv(
    explainability_csv_path,
    index=False
)


# ---------------------------------------------------------
# 6. Save explainability metadata for the application
# ---------------------------------------------------------

explainability_profile = {

    "project":
        "ClimaCare UAE AI",

    "explanation_method":
        "Permutation Importance",

    "evaluation_basis":
        (
            "External temporal validation "
            "on newer Dubai environmental data"
        ),

    "external_validation_samples":
        int(len(X_external)),

    "verified_external_r2":
        float(verification_r2),

    "interpretation":
        (
            "Higher permutation importance indicates "
            "greater predictive reliance by the model."
        ),

    "causality_warning":
        (
            "Feature importance represents predictive "
            "association and must not be interpreted "
            "as proof of causation."
        ),

    "top_prediction_drivers":
        [
            {
                "rank":
                    int(row["rank"]),

                "feature":
                    row["feature"],

                "display_name":
                    row["display_name"],

                "group":
                    row["feature_group"],

                "importance_mean":
                    float(row["importance_mean"]),

                "importance_std":
                    float(row["importance_std"])
            }

            for _, row in
            explainability_df.head(15).iterrows()
        ]
}


explainability_json_path = (
    Path("../models")
    /
    "climacare_explainability_profile.json"
)


with open(
    explainability_json_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        explainability_profile,
        file,
        indent=4
    )


# ---------------------------------------------------------
# 7. Confirmation
# ---------------------------------------------------------

print("\n✅ CLIMACARE EXPLAINABILITY PROFILE SAVED")
print("-" * 70)

print(
    "Explainability table:",
    explainability_csv_path
)

print(
    "Application profile:",
    explainability_json_path
)

print(
    "\nStrongest predictive driver:",
    explainability_df.iloc[0]["display_name"]
)

print(
    "Verified External R²:",
    round(verification_r2, 3)
)

Building ClimaCare explainability profile...

TOP 10 HUMAN-READABLE MODEL DRIVERS
----------------------------------------------------------------------


,rank,feature,display_name,feature_group,importance_mean,importance_std
0,1,pm2_5,Fine Particulate Matter (PM2.5),Air Pollution,1.2789,0.1091
1,2,ozone,Ground-Level Ozone,Air Pollution,0.0215,0.0088
2,3,us_aqi,Current Air Quality Index,Air Quality,0.0163,0.0031
3,4,temperature,Temperature,Weather,0.0095,0.0024
4,5,uv_index,UV Index,Climate / Exposure,0.0088,0.0027
5,6,pm10,Particulate Matter (PM10),Air Pollution,0.0039,0.0010
6,7,aerosol_optical_depth,Atmospheric Aerosol Load,Atmosphere,0.0032,0.0009
7,8,sulphur_dioxide,Sulphur Dioxide,Air Pollution,0.0024,0.0011
8,9,nitrogen_dioxide,Nitrogen Dioxide,Air Pollution,0.0021,0.0014
9,10,us_aqi_roll3,3-Day AQI Trend,Historical Trend,0.0020,0.0009



✅ CLIMACARE EXPLAINABILITY PROFILE SAVED
----------------------------------------------------------------------
Explainability table: ../models/climacare_global_explainability.csv
Application profile: ../models/climacare_explainability_profile.json

Strongest predictive driver: Fine Particulate Matter (PM2.5)
Verified External R²: 0.851


In [19]:
# ---------------------------------------------------------
# Step 45
# ClimaCare UAE AI
# Install and Verify SHAP Explainability Library
# ---------------------------------------------------------

%pip install shap -q

import shap

print("✅ SHAP is ready for ClimaCare local explanations.")
print("SHAP version:", shap.__version__)

Note: you may need to restart the kernel to use updated packages.
✅ SHAP is ready for ClimaCare local explanations.
SHAP version: 0.52.0


In [20]:
# ---------------------------------------------------------
# Step 46A
# Check ClimaCare Variables Before Local Explainability
# ---------------------------------------------------------

required_variables = [
    "evaluation_model",
    "X_external",
    "y_external",
    "external_dates",
    "feature_labels"
]

print("Checking ClimaCare explainability environment...")
print("-" * 55)

for variable in required_variables:
    if variable in globals():
        print(f"✅ {variable} is available")
    else:
        print(f"❌ {variable} is NOT available")

Checking ClimaCare explainability environment...
-------------------------------------------------------
✅ evaluation_model is available
✅ X_external is available
✅ y_external is available
✅ external_dates is available
✅ feature_labels is available


In [21]:
# ---------------------------------------------------------
# Step 46B
# ClimaCare UAE AI
# Recover Local Explainability Environment
# WITHOUT rerunning the full notebook
# ---------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np
import requests
import joblib
import json
import shap

print("Recovering ClimaCare explainability environment...")


# ---------------------------------------------------------
# 1. Load preserved evaluation model
# ---------------------------------------------------------

evaluation_model = joblib.load(
    "../models/climacare_evaluation_model.joblib"
)

print("✅ Evaluation model loaded")


# ---------------------------------------------------------
# 2. Load exact 28-feature schema
# ---------------------------------------------------------

with open(
    "../models/climacare_feature_schema.json",
    "r",
    encoding="utf-8"
) as file:

    feature_schema = json.load(file)

feature_columns = feature_schema["features"]

print(
    "✅ Feature schema loaded:",
    len(feature_columns),
    "features"
)


# ---------------------------------------------------------
# 3. Load saved human-readable feature labels
# ---------------------------------------------------------

saved_explainability = pd.read_csv(
    "../models/climacare_global_explainability.csv"
)

feature_labels = dict(
    zip(
        saved_explainability["feature"],
        saved_explainability["display_name"]
    )
)

print("✅ Explainability labels loaded")


# ---------------------------------------------------------
# 4. Load saved external validation results
# ---------------------------------------------------------

external_results_saved = pd.read_csv(
    "../data/processed/external_validation_results.csv",
    parse_dates=["date"]
)

print(
    "✅ External validation results loaded:",
    len(external_results_saved),
    "rows"
)


# ---------------------------------------------------------
# 5. Rebuild ONLY a small recent external window
#
# We need Aug 12-16 so that Aug 15 has:
# lag-1 + previous 3-day rolling information
# ---------------------------------------------------------

latitude = 25.07725
longitude = 55.30927

recovery_start = "2026-08-12"
recovery_end = "2026-08-16"


# ---------------------------------------------------------
# 6. Air-quality data
# ---------------------------------------------------------

air_url = (
    "https://air-quality-api.open-meteo.com/v1/air-quality"
)

air_params = {

    "latitude": latitude,
    "longitude": longitude,

    "hourly": [
        "pm10",
        "pm2_5",
        "carbon_monoxide",
        "nitrogen_dioxide",
        "sulphur_dioxide",
        "ozone",
        "aerosol_optical_depth",
        "dust",
        "uv_index",
        "us_aqi",
        "european_aqi"
    ],

    "start_date": recovery_start,
    "end_date": recovery_end,

    "timezone": "Asia/Dubai",
    "domains": "cams_global"
}

print("\nDownloading small Dubai recovery window...")

air_response = requests.get(
    air_url,
    params=air_params,
    timeout=120
)

air_response.raise_for_status()

air_hourly = pd.DataFrame(
    air_response.json()["hourly"]
)

air_hourly["time"] = pd.to_datetime(
    air_hourly["time"]
)

air_hourly["date"] = (
    air_hourly["time"].dt.normalize()
)

air_variables = [
    "pm10",
    "pm2_5",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "aerosol_optical_depth",
    "dust",
    "uv_index",
    "us_aqi",
    "european_aqi"
]

air_daily = (
    air_hourly
    .groupby("date", as_index=False)[air_variables]
    .mean()
)


# ---------------------------------------------------------
# 7. Weather data
# ---------------------------------------------------------

weather_url = (
    "https://archive-api.open-meteo.com/v1/archive"
)

weather_params = {

    "latitude": latitude,
    "longitude": longitude,

    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m"
    ],

    "start_date": recovery_start,
    "end_date": recovery_end,

    "timezone": "Asia/Dubai",
    "wind_speed_unit": "kmh"
}

weather_response = requests.get(
    weather_url,
    params=weather_params,
    timeout=120
)

weather_response.raise_for_status()

weather_hourly = pd.DataFrame(
    weather_response.json()["hourly"]
)

weather_hourly["time"] = pd.to_datetime(
    weather_hourly["time"]
)

weather_hourly["date"] = (
    weather_hourly["time"].dt.normalize()
)

weather_daily = (
    weather_hourly
    .groupby(
        "date",
        as_index=False
    )[
        [
            "temperature_2m",
            "relative_humidity_2m",
            "wind_speed_10m"
        ]
    ]
    .mean()
)

weather_daily = weather_daily.rename(
    columns={
        "temperature_2m": "temperature",
        "relative_humidity_2m": "humidity",
        "wind_speed_10m": "wind_speed"
    }
)


# ---------------------------------------------------------
# 8. Merge recovery window
# ---------------------------------------------------------

local_df = pd.merge(
    air_daily,
    weather_daily,
    on="date",
    how="inner"
)

local_df = (
    local_df
    .sort_values("date")
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 9. Recreate EXACT ClimaCare features
# ---------------------------------------------------------

local_df["year"] = local_df["date"].dt.year
local_df["month"] = local_df["date"].dt.month
local_df["day"] = local_df["date"].dt.day
local_df["day_of_week"] = local_df["date"].dt.dayofweek


def local_season(month):

    if month in [12, 1, 2]:
        return 0

    elif month in [3, 4, 5]:
        return 1

    elif month in [6, 7, 8]:
        return 2

    else:
        return 3


local_df["season_encoded"] = (
    local_df["month"].apply(local_season)
)


# Lag features
local_df["pm2_5_lag1"] = (
    local_df["pm2_5"].shift(1)
)

local_df["pm10_lag1"] = (
    local_df["pm10"].shift(1)
)

local_df["us_aqi_lag1"] = (
    local_df["us_aqi"].shift(1)
)


# Previous 3-day rolling features
local_df["pm2_5_roll3"] = (
    local_df["pm2_5"]
    .shift(1)
    .rolling(3)
    .mean()
)

local_df["pm10_roll3"] = (
    local_df["pm10"]
    .shift(1)
    .rolling(3)
    .mean()
)

local_df["us_aqi_roll3"] = (
    local_df["us_aqi"]
    .shift(1)
    .rolling(3)
    .mean()
)


# Interaction features
local_df["pm_ratio"] = (
    local_df["pm2_5"]
    /
    (local_df["pm10"] + 1e-6)
)

local_df["dust_wind"] = (
    local_df["dust"]
    /
    (local_df["wind_speed"] + 1e-6)
)

local_df["temp_humidity"] = (
    local_df["temperature"]
    *
    local_df["humidity"]
)


# Actual next-day target
local_df["target_aqi_next_day"] = (
    local_df["us_aqi"].shift(-1)
)


# ---------------------------------------------------------
# 10. Select Aug 15 for our local explanation
# ---------------------------------------------------------

local_explanation_date = pd.Timestamp(
    "2026-08-15"
)

local_row = local_df[
    local_df["date"] == local_explanation_date
].copy()


if local_row.empty:
    raise ValueError(
        "Local explanation date was not found."
    )


# Exact same feature order as training
X_local = local_row[
    feature_columns
].copy()

y_local_actual = float(
    local_row[
        "target_aqi_next_day"
    ].iloc[0]
)


# ---------------------------------------------------------
# 11. Predict
# ---------------------------------------------------------

local_prediction = float(
    evaluation_model.predict(
        X_local
    )[0]
)


# ---------------------------------------------------------
# 12. Final checks
# ---------------------------------------------------------

print("\n✅ CLIMACARE RECOVERY COMPLETED")
print("-" * 60)

print(
    "Explanation date:",
    local_explanation_date.date()
)

print(
    "Model features:",
    X_local.shape[1]
)

print(
    "Missing feature values:",
    X_local.isnull().sum().sum()
)

print(
    f"Current AQI: "
    f"{local_row['us_aqi'].iloc[0]:.2f}"
)

print(
    f"Predicted next-day AQI: "
    f"{local_prediction:.2f}"
)

print(
    f"Actual next-day AQI: "
    f"{y_local_actual:.2f}"
)

print(
    "\nReady for SHAP:",
    X_local.shape[1] == 28
    and X_local.isnull().sum().sum() == 0
)

Recovering ClimaCare explainability environment...
✅ Evaluation model loaded
✅ Feature schema loaded: 28 features
✅ Explainability labels loaded
✅ External validation results loaded: 178 rows


✅ CLIMACARE RECOVERY COMPLETED
------------------------------------------------------------
Explanation date: 2026-08-15
Model features: 28
Missing feature values: 0
Current AQI: 168.75
Predicted next-day AQI: 154.89
Actual next-day AQI: 158.96

Ready for SHAP: True


In [22]:
# ---------------------------------------------------------
# Step 47
# ClimaCare UAE AI
# Local SHAP Explanation for One Dubai Forecast
# ---------------------------------------------------------

import numpy as np
import pandas as pd
import shap

print("Generating ClimaCare local SHAP explanation...")


# ---------------------------------------------------------
# 1. Create SHAP explainer
# ---------------------------------------------------------

local_explainer = shap.TreeExplainer(
    evaluation_model
)


# ---------------------------------------------------------
# 2. Explain the selected Dubai observation
# ---------------------------------------------------------

local_shap_result = local_explainer(
    X_local
)


# ---------------------------------------------------------
# 3. Extract SHAP contributions safely
# ---------------------------------------------------------

local_shap_values = np.array(
    local_shap_result.values
).reshape(-1)

local_base_value = float(
    np.array(
        local_shap_result.base_values
    ).reshape(-1)[0]
)


# ---------------------------------------------------------
# 4. Build readable explanation table
# ---------------------------------------------------------

local_driver_df = pd.DataFrame({

    "feature":
        X_local.columns,

    "display_name":
        [
            feature_labels.get(
                feature,
                feature
            )
            for feature in X_local.columns
        ],

    "feature_value":
        X_local.iloc[0].values,

    "shap_value":
        local_shap_values
})


# Magnitude of each contribution
local_driver_df[
    "absolute_contribution"
] = np.abs(
    local_driver_df["shap_value"]
)


# Human-readable direction
def shap_effect(value):

    if value > 0:
        return "↑ Increased forecast"

    elif value < 0:
        return "↓ Reduced forecast"

    else:
        return "No meaningful change"


local_driver_df["effect"] = (
    local_driver_df["shap_value"]
    .apply(shap_effect)
)


# Sort strongest influences first
local_driver_df = (
    local_driver_df
    .sort_values(
        "absolute_contribution",
        ascending=False
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 5. Verify SHAP mathematics
# ---------------------------------------------------------

shap_reconstructed_prediction = (
    local_base_value
    +
    local_shap_values.sum()
)


print("\nCLIMACARE LOCAL FORECAST EXPLANATION")
print("-" * 65)

print(
    "Forecast date:",
    local_explanation_date.date()
)

print(
    f"Current AQI: "
    f"{X_local['us_aqi'].iloc[0]:.2f}"
)

print(
    f"Predicted next-day AQI: "
    f"{local_prediction:.2f}"
)

print(
    f"Actual next-day AQI: "
    f"{y_local_actual:.2f}"
)

print(
    f"Prediction error: "
    f"{abs(y_local_actual - local_prediction):.2f}"
)


print("\nSHAP VALIDATION")
print("-" * 65)

print(
    f"Model baseline value: "
    f"{local_base_value:.2f}"
)

print(
    f"SHAP reconstructed prediction: "
    f"{shap_reconstructed_prediction:.2f}"
)

print(
    f"Actual model prediction: "
    f"{local_prediction:.2f}"
)


# ---------------------------------------------------------
# 6. Top local prediction drivers
# ---------------------------------------------------------

print("\nTOP 10 FACTORS INFLUENCING THIS FORECAST")
print("-" * 65)

display(
    local_driver_df[
        [
            "display_name",
            "feature_value",
            "shap_value",
            "effect"
        ]
    ]
    .head(10)
    .round(3)
)

Generating ClimaCare local SHAP explanation...

CLIMACARE LOCAL FORECAST EXPLANATION
-----------------------------------------------------------------
Forecast date: 2026-08-15
Current AQI: 168.75
Predicted next-day AQI: 154.89
Actual next-day AQI: 158.96
Prediction error: 4.07

SHAP VALIDATION
-----------------------------------------------------------------
Model baseline value: 114.32
SHAP reconstructed prediction: 154.89
Actual model prediction: 154.89

TOP 10 FACTORS INFLUENCING THIS FORECAST
-----------------------------------------------------------------


,display_name,feature_value,shap_value,effect
0,Fine Particulate Matter (PM2.5),74.450,39.438,↑ Increased forecast
1,Ground-Level Ozone,80.625,-2.482,↓ Reduced forecast
2,Current Air Quality Index,168.750,1.479,↑ Increased forecast
3,Particulate Matter (PM10),108.025,-0.735,↓ Reduced forecast
4,Temperature,35.542,0.664,↑ Increased forecast
5,3-Day AQI Trend,171.139,0.628,↑ Increased forecast
6,UV Index,2.148,0.611,↑ Increased forecast
7,3-Day PM10 Trend,142.281,0.537,↑ Increased forecast
8,3-Day PM2.5 Trend,85.794,-0.516,↓ Reduced forecast
9,Temperature-Humidity Interaction,2222.835,0.407,↑ Increased forecast


In [23]:
# ---------------------------------------------------------
# Step 48
# ClimaCare UAE AI
# User-Friendly Forecast Explanation Engine
# ---------------------------------------------------------

import json
from pathlib import Path
import numpy as np

print("Building ClimaCare user-facing explanation...")


# ---------------------------------------------------------
# 1. AQI category helper
# ---------------------------------------------------------

def get_aqi_category(aqi):

    if aqi <= 50:
        return "Good"

    elif aqi <= 100:
        return "Moderate"

    elif aqi <= 150:
        return "Unhealthy for Sensitive Groups"

    elif aqi <= 200:
        return "Unhealthy"

    elif aqi <= 300:
        return "Very Unhealthy"

    else:
        return "Hazardous"


# ---------------------------------------------------------
# 2. Separate upward and downward model drivers
# ---------------------------------------------------------

increasing_drivers = (
    local_driver_df[
        local_driver_df["shap_value"] > 0
    ]
    .sort_values(
        "absolute_contribution",
        ascending=False
    )
    .head(5)
)

reducing_drivers = (
    local_driver_df[
        local_driver_df["shap_value"] < 0
    ]
    .sort_values(
        "absolute_contribution",
        ascending=False
    )
    .head(5)
)


# ---------------------------------------------------------
# 3. Convert drivers into app-friendly objects
# ---------------------------------------------------------

def prepare_driver_records(driver_df, direction):

    records = []

    for _, row in driver_df.iterrows():

        records.append({

            "name":
                row["display_name"],

            "observed_value":
                round(
                    float(row["feature_value"]),
                    2
                ),

            "model_contribution":
                round(
                    float(row["shap_value"]),
                    2
                ),

            "direction":
                direction
        })

    return records


upward_records = prepare_driver_records(
    increasing_drivers,
    "increased forecast"
)

downward_records = prepare_driver_records(
    reducing_drivers,
    "reduced forecast"
)


# ---------------------------------------------------------
# 4. Create concise explanation sentence
# ---------------------------------------------------------

top_up_names = [
    item["name"]
    for item in upward_records[:3]
]

top_down_names = [
    item["name"]
    for item in downward_records[:2]
]


if top_up_names:

    upward_summary = (
        "The strongest signals pushing the forecast "
        "higher were "
        + ", ".join(top_up_names)
        + "."
    )

else:

    upward_summary = (
        "No major upward-driving signal was identified."
    )


if top_down_names:

    downward_summary = (
        "The strongest signals pulling the forecast "
        "lower were "
        + ", ".join(top_down_names)
        + "."
    )

else:

    downward_summary = (
        "No major downward-driving signal was identified."
    )


# ---------------------------------------------------------
# 5. ClimaCare early-warning status
# ---------------------------------------------------------

CLIMACARE_WARNING_THRESHOLD = 90

early_warning_active = (
    local_prediction >
    CLIMACARE_WARNING_THRESHOLD
)


# ---------------------------------------------------------
# 6. Build complete application explanation payload
# ---------------------------------------------------------

local_explanation_payload = {

    "project":
        "ClimaCare UAE AI",

    "location":
        "Dubai, UAE",

    "forecast_date":
        str(
            local_explanation_date.date()
        ),

    "current_aqi":
        round(
            float(X_local["us_aqi"].iloc[0]),
            2
        ),

    "predicted_next_day_aqi":
        round(
            float(local_prediction),
            2
        ),

    "predicted_category":
        get_aqi_category(
            local_prediction
        ),

    "early_warning": {

        "active":
            bool(early_warning_active),

        "prediction_trigger":
            CLIMACARE_WARNING_THRESHOLD,

        "note":
            (
                "ClimaCare's early-warning trigger is "
                "separate from the official AQI category."
            )
    },

    "explanation": {

        "headline":
            (
                "Why did ClimaCare produce "
                "this forecast?"
            ),

        "upward_summary":
            upward_summary,

        "downward_summary":
            downward_summary,

        "increasing_drivers":
            upward_records,

        "reducing_drivers":
            downward_records,

        "interpretation_note":
            (
                "These values explain the model's "
                "prediction and do not prove causation."
            )
    }
}


# ---------------------------------------------------------
# 7. Display clean summary
# ---------------------------------------------------------

print("\nCLIMACARE USER EXPLANATION")
print("-" * 70)

print(
    "Location:",
    local_explanation_payload["location"]
)

print(
    "Forecast date:",
    local_explanation_payload["forecast_date"]
)

print(
    "Predicted next-day AQI:",
    local_explanation_payload[
        "predicted_next_day_aqi"
    ]
)

print(
    "Predicted category:",
    local_explanation_payload[
        "predicted_category"
    ]
)

print(
    "Early warning:",
    local_explanation_payload[
        "early_warning"
    ]["active"]
)

print("\nWhy this forecast?")
print(upward_summary)

print(downward_summary)


# ---------------------------------------------------------
# 8. Save example payload for future app development
# ---------------------------------------------------------

sample_explanation_path = (
    Path("../data/processed")
    /
    "climacare_sample_explanation.json"
)


with open(
    sample_explanation_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        local_explanation_payload,
        file,
        indent=4
    )


print(
    "\n✅ Sample app explanation saved:"
)

print(sample_explanation_path)

Building ClimaCare user-facing explanation...

CLIMACARE USER EXPLANATION
----------------------------------------------------------------------
Location: Dubai, UAE
Forecast date: 2026-08-15
Predicted next-day AQI: 154.89
Predicted category: Unhealthy
Early warning: True

Why this forecast?
The strongest signals pushing the forecast higher were Fine Particulate Matter (PM2.5), Current Air Quality Index, Temperature.
The strongest signals pulling the forecast lower were Ground-Level Ozone, Particulate Matter (PM10).

✅ Sample app explanation saved:
../data/processed/climacare_sample_explanation.json
